In [128]:
import numpy as np
import pandas as pd
import random
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sc
import math
from scipy import stats 
from scipy.optimize import minimize
import statsmodels.formula.api as smf
import statsmodels as sm

In [129]:
np.random.seed(42)
random.seed(42)

Loooong introduction

Let's conduct a simple AB-test on simulated data, where:
$$\alpha=0.05,\beta=0.2\:(Type\:I,II\:error\:rates)$$
$$ x^c_1,x^c_2,\cdots,x_{n_c}^c \sim iid\:Bern(p_c) \:(control\: group) $$
$$ x^t_1,x^t_2,\cdots,x_{n_t}^t \sim iid\:Bern(p_t) \:(treatment\: group) $$

For example we want to improve conversion rate by 2%, then MDE=0.02 and we need to check that:
$$ H_0:p_c=p_t $$
$$ H_a:p_t \ne p_c $$

Then we need to calculate the number of observations we need and proportion between sample sizes of groups. As proportion q we can set:

$$ 1)\:q=0.5 $$
$$ 2)\:conduct \:2\:tests\:with\:q=0.99\:and\:q=0.01 $$
$$ 3)\:select\:q\:to\:minimize\:overall\:variance $$
$$ 4)\:find \: and \: use \: any \: different \: method \: :) $$

Next step is to calculate the number of observations we need, we can do it analytically or conduct a simulation. Analytical solution :
$$ Let's\:say\:that\:d-distribution\:of\:merged\:groups, \widehat{d}-proportion $$
$$ \beta=P(H_0 \:|\: H_a)=P(\frac{\widehat{d}}{se(\widehat{d})} \le Z_{1-\alpha} \: | \: H_a)=P(\widehat{d} \le Z_{1-\alpha}\cdot{se(\widehat{d})} \:|\: H_a) =  $$
$$ P(\frac{\widehat{d}-MDE}{se(\widehat{d})} \le \frac{Z_{1-\alpha} \cdot se(\widehat{d}) - MDE}{se(\widehat{d})}) = P(N(0;1) \le X)\: (denote \:right\:part \: of\:inequality\:as \:X) $$
$$ We\:can\:notice\:that\: \beta=F_{N(0;1)}(X)\: (cdf \: of\:(0;1)\:normal\:curve) $$
$$ By \: applying \: F^{-1} \: to\:both\:sides\:of\:\beta=F_{N(0;1)}(X)\to Z_{\beta}=X $$
$$ -Z_{1-\beta}=Z_{\beta} \:(symmetry\:of\:normal\:curve) $$
$$ From\:substitution\:se(\widehat{d})=\frac{MDE}{Z_{1-\alpha}+Z_{1-\beta}},\:se(\widehat{d})^2=\frac{\widehat{\sigma_c}^2}{q\cdot n}+\frac{\widehat{\sigma_t}^2}{(1-q)\cdot n}$$

In [130]:
def obs_num(alpha, beta, mde, sigmat, sigmac, q):
    var_c = sigmac ** 2 / q
    var_t = sigmat ** 2 / (1 - q)
    ratio = (sc.stats.norm.ppf(1 - alpha) + sc.stats.norm.ppf(1-beta)) ** 2 / mde ** 2
    return (var_c + var_t) * ratio

Instead of plugging q=0.5, let's find q that minimizes the variance 

$$ var(\widehat{d})=\frac{\widehat{\sigma_c}^2}{q\cdot n}+\frac{\widehat{\sigma_t}^2}{(1-q)\cdot n}, \:\frac{d (var(\widehat{d}))}{dq}=0,\:q=\frac{\widehat{\sigma_t}}{\widehat{\sigma_t} +\widehat{\sigma_c}} $$

Hold on, now we can actually start testing stuff :)

In [131]:
#Initial parameters
alpha = 0.05
beta = 0.2
mde = 0.02

#Simulation of data
DAU = 100
active = DAU + np.random.normal(loc=15, scale=6, size=7).astype(int) #Number of users for each day
sample = np.random.binomial(p=0.017, n=1, size=active.sum()) #Conversion of users

#Estimation of p_c from historical data
p_c = sample.mean()
p_c

0.01701093560145808

In [132]:
#theoretical proportion of control group
p_t = p_c + 0.02

#var, std
var_t = p_t * (1 - p_t)
var_c = p_c * (1 - p_c)
s_t = np.sqrt(var_t)
s_c = np.sqrt(var_c)

#Proportion of split
q = s_t / (s_t + s_c)
n = obs_num(alpha, beta, mde, s_t, s_c, q)
q, n

(0.5934872745961782, 1790.6293718682525)

If we can't collect necessary number of observation within a given period of time, we can try to reduce variance or MDE. Second option is to increase Type I, II error rates (alpha and beta respectively)

In [133]:
DAU = 270
active = DAU + np.random.normal(loc=15, scale=6, size=7).astype(int)
sample_c = np.random.binomial(p=0.017, n=1, size= int(active.sum() * q) + 1 ) #Control
sample_t = np.random.binomial(p=0.017 + mde + 0.002, n=1, size= int(active.sum() * (1 - q)) + 1) #Treatment

len(sample_c), len(sample_t), len(sample_c) + len(sample_t)

(1180, 808, 1988)

In [134]:
p_c = sample_c.mean()
p_t = sample_t.mean()
var_c = p_c * (1 - p_c) / len(sample_c)
var_t = p_t * (1 - p_t) / len(sample_t)
se_d = np.sqrt(var_c + var_t)

ci = sc.stats.norm.interval(loc=p_t - p_c, scale=se_d, confidence=alpha)
print(ci)

(0.022374862902638935, 0.023446583648795868)


Our 95%-CI doesn't include 0 so H0 is not supported.

In [135]:
count = 0
for j in range(3):
    count = 0
    for i in range(2000):
        n = 1810
        sample_c = np.random.binomial(p=0.017, n=1, size=int(n * q) + 1)
        sample_t = np.random.binomial(p=0.017 + mde + 0.002, n=1, size=int(n * (1 - q)) + 1)

        p_c = sample_c.mean()
        p_t = sample_t.mean()

        var_c = p_c * (1 - p_c) / len(sample_c)
        var_t = p_t * (1 - p_t) / len(sample_t)
        se_d = np.sqrt(var_c + var_t)

        ci = sc.stats.norm.interval(loc=p_t - p_c, scale=se_d, confidence=1 - alpha)
        count += 1 if ci[0] > 0 else 0

    print("Power (1 - Beta):", count/2000)


Power (1 - Beta): 0.806
Power (1 - Beta): 0.7845
Power (1 - Beta): 0.7785


In [ ]:
count = 0
for j in range(3):
    count = 0
    for i in range(2000):
        n = 1810
        sample_c = np.random.binomial(p=0.017, n=1, size=int(n * q) + 1)
        sample_t = np.random.binomial(p=0.017 + 0.001, n=1, size=int(n * (1 - q)) + 1)

        p_c = sample_c.mean()
        p_t = sample_t.mean()

        var_c = p_c * (1 - p_c) / len(sample_c)
        var_t = p_t * (1 - p_t) / len(sample_t)
        se_d = np.sqrt(var_c + var_t)

        ci = sc.stats.norm.interval(loc=p_t - p_c, scale=se_d, confidence=1 - alpha)
        count += 1 if ci[0] > 0 else 0

    print("Alpha:", count/2000)

Alpha: 0.028
Alpha: 0.0365
Alpha: 0.0355


Power is around 80% while alpha doesn't exceed the established boundaries which is good